# Port Doxygen comment blocks to program specific documentation format

The goal of this notebook is to enable user to rapidly generate documentation for programming languages which effectively "wrap" the falcon-core C++ code base. This is necessary for creating auto-tunning algorithmic builds on-top-of a desired programming language.

Current wrapper support is for:
- C 

For demonstration purposes, this notebook outlines how to map C++ documentation to C via the following Python scripts: 

1) `/docs_managment_code/upgrade_doxygen_comments.py`: Clean-up comment blocks to ensure doxygen formatting,
2) `/docs_managment_code/extract_cpp_docs.py`: Extract doxygen comment blocks and temporarily store in a cpp_metadata directory,
3)  `/docs_managment_code/generate_c_api_maps.py`: Construct cpp code mapping configuration yaml files for each cpp file,
4) `/docs_managment_code/inject_c_docs.py`: Inject cpp comment blocks in desired programming language code,
5) Helper scripts.


However, the make file `capi_docs.mk` contains all of the necessary functionality for maintaining the c-api documentation and has been included in the `Makefile`.

Commands:`make docs-all`, `docs-setup`, `docs-run`, `docs-coverage`, `docs-teardown`, `clean_logs`

`make docs-all`: docs-setup docs-run docs-coverage


## 1. Clean C++ Comment Blocks

This script updates any `/* ... strings ... */` to `/** ... strings ... */`, which are Doxygen formatted C++ comment blocks.

In [ ]:
!python3 ./docs_managment_code/upgrade_doxygen_comments.py ./cpp/include --write --verbose

## 2. Extract C++ Comment Blocks to Metadata Directory

This script scrapes the desired target directory of all Doxygen formatted strings and dumps them into a library for future reference.

**NOTE:** The is no inherent reason to save this library, but it is instructive to have for debugging purposes.

In [ ]:
# 1) Extract metadata into docs_managment_code/cpp_metadata
!python3 docs_managment_code/extract_cpp_docs.py \
  ./cpp/include \
  # --meta-root ./docs_managment_code/cpp_metadata >> ./docs_managment_code/logs/extract_cpp_docs.log

## 3. Generate Configuration Files that Map C++ Documentation to the Desired Programming Language

This script automates the write of .map.yml files into the target language directory (in this example C code). Current expectation is that the C code maps to C++ code via `<C++ Class Name>_<Class methhod>`.

In [ ]:
# 2) Generate auto maps
!python3 docs_managment_code/generate_c_api_maps.py \
  --cpp-metadata-root ./cpp_metadata \
  --cpp-include-root  ./cpp/include \
  --c-api-root        ./c-api/include \
  --overwrite \
  --verbose >> ./docs_managment_code/logs/cpp_auto_mapping_log.txt

In [ ]:
# # 2) Generate auto maps
# !python3 docs_managment_code/generate_c_api_maps.py \
#   --cpp-metadata-root ./docs_managment_code/cpp_metadata \
#   --cpp-include-root  ./cpp/include \
#   --c-api-root        ./c-api/include \
#   --overwrite \
#   --verbose >> ./docs_managment_code/logs/cpp_auto_mapping_log.txt

## 4. Inject Doxygen Formatted Documentation

This script inserts Doxygen comment blocks into the desired language, given a .map.yml mapping.

**NOTE:** This current script assumes Doxygen injection to C code. If a different language is desired, then a new injector must be created.

In [ ]:
!python3 ./docs_managment_code/inject_c_docs.py \
  --capi-root ./c-api \
  --cpp-root ./cpp \
  --cpp-metadata-root ./docs_managment_code/cpp_metadata \
  --maps-dir ./c-api/include/falcon_core \
  --user-maps-dir ./docs_managment_code/c-api_user_maps \
  --out-root ./c-api \
  --verbose >> ./docs_managment_code/logs/inject_c_docs.log


In [ ]:
!python3 ./docs_managment_code/inject_c_docs.bak1.py \
  --capi-root ./c-api \
  --cpp-root ./cpp \
  --cpp-metadata-root ./cpp_metadata \
  --maps-dir ./c-api/include/falcon_core \
  --out-root ./c-api \
  --verbose >> ./docs_managment_code/logs/inject_c_docs.log


## 5. Helper Scripts

This first script allows one to determine the number of classes/functions that:
- Have Doxygen comment blocks
- Total number of class methods or stand-alone functions

**NOTE:** This script does not compare coverage from C++ to the desired language; simple the coverage in a desired language.

In [ ]:
# !python3 ./docs_managment_code/doxygen_coverage.py ./cpp/include --per-file >> ./docs_managment_code/logs/cpp_coverage_report.txt

In [ ]:
# !python3 ./docs_managment_code/doxygen_coverage.py ./c-api/include --per-file >> ./docs_managment_code/logs/capi_coverage_report.txt

This script compares the coverage between C++ and the desired language.

**NOTE:** This assumes the target language is C.

In [ ]:
!python3 ./docs_managment_code/doxygen_port_coverage.py \
    --cpp-root           ./cpp/include \
    --cpp-metadata-root  ./cpp_metadata \
    --c-root             ./c-api/include